<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Recommender Systems - Solution

---

### About
Define what a recommender system is, when it is used, and build one.

### Learning Objectives
- Explain the difference between collaborative filtering and content-based systems.
- Understand Cosine Similarity.
- Explain the math behind recommendation engines.
- Create an item-based movie recommender.

### Notebook Guide
- Imports
- Load and Process Data
- Now You
- Create pivot table
- Create sparse matrix
- Calculate cosine similarity
- Create distances DataFrame
- Time to Play!
- Conclusions and Takeaways

# Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import pairwise_distances

# Load and Process Data
---

We'll be using the [MovieLens](https://grouplens.org/datasets/movielens/) data set for building our recommendation engine. There are two CSVs (`movies.csv` and `ratings.csv`) that we'll eventually inner join.

In [ ]:
movies = pd.read_csv('../data/movies.csv')
movies.head()

In [ ]:
movies.info()

In [ ]:
ratings = pd.read_csv('../data/ratings.csv')
ratings.head()

In [ ]:
ratings.info()

# Now You
---

## Merge 'em!

In [ ]:
# YOU: Merge these data sets. What kind of merge will it be?
df = pd.merge(ratings, movies, on='movieId', how='left')
df.head()

In [ ]:
# Always after a merge, make sure the row counts make sense!
print(ratings.shape)
print(movies.shape)
print(df.shape)

## Drop 'em!
We really only need the user IDs, ratings, and movie title. Drop everything else.

In [ ]:
df.drop(columns=['timestamp', 'movieId', 'genres'], inplace=True)

In [ ]:
df.head()

# Create pivot table
---

Because we're creating an item-based collaborative recommender (where item in this case is our movies), we'll set up our pivot table as follows:
1. The `title` will be the index
2. The `userId` will be the column
3. The `rating` will be the value

**If we were building a user-based collaborative recommender, what would change about this pivot table?**

In [ ]:
rating_mx = pd.pivot_table(
    df,
    index='title',
    columns='userId',
    values='rating'
)

rating_mx.head()

# Optionally mean "center"
#  - Note that broadcasting an ndarray onto a 2D matrix does not work
#     automatically with a Series, hence must reshape to a col vector
# rating_mx - 3

In [ ]:
rating_mx.shape

# Create sparse matrix
---

In a minute, we'll calculate the cosine similarity for each movie using the `pairwise_distances` function. Before that, we need to create a sparse matrix (datatype) using `scipy`'s `sparse` module like so:
```python
sparse.csr_matrix(pivot.fillna(0), metric='cosine')
```

In [ ]:
ratings_sparse = sparse.csr_matrix(rating_mx.fillna(0))
# print(ratings_sparse)

# Calculate cosine similarity
---

`sklearn` has a built-in `pairwise_distances` function that we can use for our recommender. It will return a square matrix, comparing every movie with every other movie in the data set.

```python
pairwise_distances(sparse_pivot, metric='cosine')
cosine_distances(sparse_pivot)                     # Identical but more concise
```

In [ ]:
# Note that a distance of 1 is a similarity of 0.
dists = pairwise_distances(ratings_sparse, metric='cosine')
# Identical but more concise
# dists = cosine_distances(ratings_sparse)

np.round(dists[:10, :10], 2)

In [ ]:
dists.shape

# Create distances DataFrame
---
At this point, we essentially have a recommender! We'll load it into a `pandas` DataFrame for ease-of-use. 

In [ ]:
df_rec = pd.DataFrame(dists, columns=rating_mx.index, index=rating_mx.index)
df_rec.iloc[:5, :5]

# Time to Play!
We've got our recommender - let's try out a few movies.

In [ ]:
titles = rating_mx.index
titles[titles.str.lower().str.contains('lord of the rings')]

In [ ]:
df_rec['Lord of the Rings: The Fellowship of the Ring, The (2001)'].sort_values().head()

In [ ]:
df_rec['John Wick (2014)'].sort_values().head()

In [ ]:
df_rec['Shrek (2001)'].sort_values().head()

In [ ]:
# What the hell?
df_rec['Frozen (2010)'].sort_values().head()

# Conclusions and Takeaways
---
Recommender systems are cool, and there are lots of ways to create them! Which technique you pick is often determined by what kind of rating data (if any) you have.

### Types of Recommenders
1. Collaborative filtering (what we did today!)
1. Content-based recommenders (use information about the items themselves)
1. Knowledge-based recommenders (use user-input to make informed decisions)
1. Hybrid or ensemble recommenders (some combination of 1-3 above)